# 11.3 Анализ A/B-теста — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 11 «A/B-тестирование» · страница 11.3.

В ноутбуке три блока кода страницы на pandas: проверка групп и оценка эффектов, дельта-метод для отношений сумм и бутстрэп по пользователям. Первый блок страницы — строка на пользователя на SQL — лежит рядом файлом `checkout_v2_users.sql`, ему нужен PostgreSQL. Шаблон отчёта — в `report_template.md`. Данные учебные: эксперимент checkout_v2 из общего набора темы `data/ab/`.

**Как работать.** Выполняйте ячейки по порядку сверху вниз: каждый блок продолжает предыдущий. Последняя ячейка выполняет 2 000 повторных выборок.

Вывод получен на Python 3.13, pandas 3.0, NumPy 2.5, SciPy 1.18 и statsmodels 0.15.

## Где лежат данные

Код страницы читает файлы по пути от корня папки с материалами: `data/ab/sessions.parquet` и `data/ab/exposures.csv`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd
import scipy
import statsmodels

root = pathlib.Path.cwd()
while not (root / "data" / "ab").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/ab/
assert (root / "data" / "ab" / "sessions.parquet").exists(), \
    "Не найдена папка data/ab/. Скачайте общий набор темы 11 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| NumPy", np.__version__,
      "| SciPy", scipy.__version__, "| statsmodels", statsmodels.__version__)

Python 3.13.13 | pandas 3.0.5 | NumPy 2.5.3 | SciPy 1.18.1 | statsmodels 0.15.0


## Подготовка данных и оценка эффектов

### От лога до отчёта: баланс, тесты, интервалы

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

s = pd.read_parquet("data/ab/sessions.parquet").query("experiment == 'checkout_v2'")
x = pd.read_csv("data/ab/exposures.csv").query("experiment == 'checkout_v2'")
Z95 = stats.norm.ppf(0.975)
fmt = lambda v, d=2, sign="": f"{v:{sign}.{d}f}".replace(".", ",").replace("-", "−")   # 0,59 вместо 0.59

# 1) баланс до метрик: сколько людей получила каждая группа против задуманных 50/50
n = x.group.value_counts().sort_index()
p_srm = stats.chisquare(n).pvalue
print(f"показов: A {n.A}, B {n.B}; доля B {n.B / n.sum():.2%}; χ² против 50/50: p = {p_srm:.2f}")

# 2) лог → строка на пользователя; группа — из показа (хеш), а не из столбца group сессии
u = (s.groupby("user_id")
      .agg(sessions=("session_id", "size"), conv=("converted", "sum"),
           orders=("orders", "sum"), revenue=("revenue", "sum"))
      .join(x.set_index("user_id").group, how="right", validate="one_to_one")
      .fillna({"sessions": 0, "conv": 0, "orders": 0, "revenue": 0}).sort_index())
both = (s.groupby("user_id").group.nunique() > 1).sum()   # сессии записаны в обеих группах
bots = u.query("sessions > 200 and orders == 0").index     # сотни сессий без единого заказа
u = u.drop(bots)
u["buyer"] = (u.orders > 0).astype(int)
print(f"сессий {len(s)} → пользователей {len(u)}; ботов убрано {len(bots)}; в обеих группах {both}")


def mean_ci(col):
    """Средние A и B, разница с интервалом и относительный эффект с интервалом (дельта-метод)."""
    a, b = u.loc[u.group == "A", col], u.loc[u.group == "B", col]
    va, vb = a.var() / len(a), b.var() / len(b)
    d, se = b.mean() - a.mean(), np.sqrt(va + vb)
    r = b.mean() / a.mean()
    se_r = r * np.sqrt(va / a.mean()**2 + vb / b.mean()**2)
    p = stats.ttest_ind(b, a, equal_var=False).pvalue        # Уэлч
    return a.mean(), b.mean(), d, d - Z95 * se, d + Z95 * se, r - 1, r - 1 - Z95 * se_r, r - 1 + Z95 * se_r, p


# 3) доля покупателей: z-тест и интервал разницы (statsmodels), относительный эффект
k, m = u.groupby("group").buyer.sum(), u.groupby("group").size()
z, p_conv = proportions_ztest([k.B, k.A], [m.B, m.A])
lo, hi = confint_proportions_2indep(k.B, m.B, k.A, m.A, method="wald")
ca, cb, dc, _, _, rc, rlo, rhi, _ = mean_ci("buyer")
# 4) выручка на человека: t-тест Уэлча, при таких n интервал — нормальное приближение
ra, rb, dr, dlo, dhi, rr, rrlo, rrhi, p_rev = mean_ci("revenue")

print(f"""
ОТЧЁТ · checkout_v2 · новое оформление заказа · 7 сен — 4 окт 2026
Единица — пользователь ({len(u)}), рандомизация по хешу user_id
Качество: B {fmt(100 * n.B / n.sum())}% показов (p = {fmt(p_srm)}), ботов {len(bots)}, в обеих группах {both} — по группе показа
Доля покупателей: {fmt(100 * ca)} → {fmt(100 * cb)}%
  {fmt(100 * dc, 2, "+")} п. п. [{fmt(100 * lo)}; {fmt(100 * hi)}], {fmt(100 * rc, 1, "+")}% [{fmt(100 * rlo, 1)}; {fmt(100 * rhi, 1)}]; z = {fmt(z)}, p = {fmt(p_conv, 5)}
Выручка на человека: {fmt(ra, 1)} → {fmt(rb, 1)} ₽
  {fmt(dr, 1, "+")} ₽ [{fmt(dlo, 1)}; {fmt(dhi, 1)}], {fmt(100 * rr, 1, "+")}% [{fmt(100 * rrlo, 1)}; {fmt(100 * rrhi, 1)}]; p = {fmt(p_rev)}
Интервалы — 95%, относительные — дельта-методом""")

показов: A 28440, B 28360; доля B 49.93%; χ² против 50/50: p = 0.74
сессий 299924 → пользователей 56797; ботов убрано 3; в обеих группах 331



ОТЧЁТ · checkout_v2 · новое оформление заказа · 7 сен — 4 окт 2026
Единица — пользователь (56797), рандомизация по хешу user_id
Качество: B 49,93% показов (p = 0,74), ботов 3, в обеих группах 331 — по группе показа
Доля покупателей: 14,08 → 15,25%
  +1,17 п. п. [0,59; 1,75], +8,3% [4,0; 12,6]; z = 3,95, p = 0,00008
Выручка на человека: 998,9 → 1076,8 ₽
  +77,9 ₽ [−37,1; 192,8], +7,8% [−4,2; 19,8]; p = 0,18
Интервалы — 95%, относительные — дельта-методом


pandas 3.0.5, SciPy 1.18.1, statsmodels 0.15.0; файлы — из data/ab. `confint_proportions_2indep` с `method="wald"` — тот же нормальный интервал, что в формуле страницы; z-тест считает ошибку по общей доле, интервал — по долям групп.

## Дельта-метод для отношений сумм

### Дельта-метод для ratio-метрик

In [3]:
# продолжение: u — строка на пользователя, s — лог сессий, x — показы, bots — боты
def ratio_delta(X, Y):
    """Отношение сумм ΣX / ΣY и его дисперсия по дельта-методу; строка (X, Y) — независимая единица."""
    X, Y = np.asarray(X, float), np.asarray(Y, float)
    n, r = len(X), X.sum() / Y.sum()
    var = (X.var(ddof=1) - 2 * r * np.cov(X, Y)[0, 1] + r**2 * Y.var(ddof=1)) / (n * Y.mean()**2)
    return r, var


# та же формула для конверсии по сессиям: у каждой сессии знаменатель 1, и формула превращается в наивную p(1 − p)/n
sess = (s[~s.user_id.isin(bots)].drop(columns="group")
        .merge(x[["user_id", "group"]], on="user_id")
        .rename(columns={"converted": "conv"}).assign(sessions=1))

rows = []
for name, num, den in [("конверсия сессий", "conv", "sessions"), ("средний чек, ₽", "revenue", "orders")]:
    se = {}
    for unit, t in [("сессия", sess), ("человек", u)]:
        (ra, va), (rb, vb) = (ratio_delta(t.loc[t.group == g, num], t.loc[t.group == g, den]) for g in "AB")
        se[unit] = np.sqrt(va + vb)
    rows.append([name, ra, rb, rb - ra, se["сессия"], se["человек"], se["человек"] / se["сессия"]])
tab = pd.DataFrame(rows, columns=["метрика", "A", "B", "B − A", "ошибка: сессия", "ошибка: человек", "шире в"])
print(tab.set_index("метрика").T.round(5).to_string())

d, (sn, sh) = rows[0][3], rows[0][4:6]
print(f"\nконверсия сессий, 95% интервал разницы, п. п.:"
      f"\n  наивно   [{100 * (d - Z95 * sn):.3f}; {100 * (d + Z95 * sn):.3f}], z = {d / sn:.2f}"
      f"\n  по людям [{100 * (d - Z95 * sh):.3f}; {100 * (d + Z95 * sh):.3f}], z = {d / sh:.2f}")

метрика          конверсия сессий  средний чек, ₽
A                         0.03730      4794.61097
B                         0.04117      4740.82596
B − A                     0.00387       -53.78501
ошибка: сессия            0.00071       152.25602
ошибка: человек           0.00090       227.07296
шире в                    1.27328         1.49139

конверсия сессий, 95% интервал разницы, п. п.:
  наивно   [0.248; 0.526], z = 5.45
  по людям [0.210; 0.564], z = 4.28


Ячейка продолжает предыдущую. Для конверсии при расчёте по сессиям знаменатель каждой строки равен 1, ковариация и дисперсия знаменателя обнуляются, и формула даёт наивную ошибку z-теста.

## Бутстрэп по пользователям и ограничение крупных значений

Зерно генератора зафиксировано, 2 000 перевыборок на группу: повторный запуск даёт те же числа.

### Бутстрэп по пользователям и обрезка хвоста

In [4]:
# продолжение: бутстрэп — перевыбираем строки с возвращением внутри каждой группы
rng = np.random.default_rng(113)


def boot_diff(t, num, den, B=2000):
    """B пересчётов ΣX/ΣY в B минус то же в A; строка t — единица перевыборки."""
    out = []
    for g in "AB":
        X, Y = t.loc[t.group == g, num].to_numpy(), t.loc[t.group == g, den].to_numpy()
        idx = (rng.integers(0, len(X), len(X)) for _ in range(B))
        out.append(np.array([X[i].sum() / Y[i].sum() for i in idx]))
    return out[1] - out[0]


d_user = boot_diff(u, "conv", "sessions")        # человек со всеми его сессиями
d_sess = boot_diff(sess, "conv", "sessions")     # сессии по одной: повторяет наивную ошибку
lo, hi = np.percentile(d_user, [2.5, 97.5])
print(f"конверсия сессий, ошибка разницы: по людям {d_user.std(ddof=1):.5f}, по сессиям {d_sess.std(ddof=1):.5f}")
print(f"  95% интервал по людям, п. п.: [{100 * lo:.3f}; {100 * hi:.3f}]")

# выручка на человека: среднее = сумма выручки / число людей
d_rev = boot_diff(u.assign(one=1), "revenue", "one")
lo, hi = np.percentile(d_rev, [2.5, 97.5])
print(f"выручка на человека, бутстрэп: разница {u.groupby('group').revenue.mean().diff().iloc[-1]:.1f} ₽, "
      f"95% интервал [{lo:.1f}; {hi:.1f}] ₽")

# обрезка хвоста: выручку каждого участника ограничиваем общим 99-м процентилем
cap = u.revenue.quantile(0.99)
w = u.assign(revenue=u.revenue.clip(upper=cap))
a, b = w.loc[w.group == "A", "revenue"], w.loc[w.group == "B", "revenue"]
d, se = b.mean() - a.mean(), np.sqrt(a.var() / len(a) + b.var() / len(b))
print(f"обрезка по {cap:.0f} ₽: {a.mean():.1f} → {b.mean():.1f} ₽, разница {d:.1f} ₽ "
      f"[{d - Z95 * se:.1f}; {d + Z95 * se:.1f}]; p = {stats.ttest_ind(b, a, equal_var=False).pvalue:.4f}")
print(f"участников с выручкой выше порога: {(u.revenue > cap).sum()}, у них {u.revenue[u.revenue > cap].sum() / u.revenue.sum():.0%} выручки")

конверсия сессий, ошибка разницы: по людям 0.00091, по сессиям 0.00071
  95% интервал по людям, п. п.: [0.206; 0.573]


выручка на человека, бутстрэп: разница 77.9 ₽, 95% интервал [-36.0; 196.2] ₽
обрезка по 14950 ₽: 744.4 → 813.4 ₽, разница 69.1 ₽ [29.3; 108.8]; p = 0.0007
участников с выручкой выше порога: 567, у них 39% выручки


Ячейка продолжает предыдущие. Зерно генератора зафиксировано, 2 000 перевыборок на группу; повторный запуск даёт те же числа. Время выполнения зависит от компьютера.